# This is my first submission on kaggle, please upvote if you find it useful.

# Hi
<img src="https://media.giphy.com/media/WpIPS0DWNpMm4kfMVr/giphy.gif">

## Titanic Passengers Survival Prediction

We will build a predictive model that answers the question: “what sorts of people were more likely to survive?” using passenger data (ie name, age, gender, socio-economic class, etc) step by step.

We'll cover the followings:

1. Problem understanding and definition

2. Data collection and preparation
   * 2.1: Loading the data files
   * 2.2: Data Description
   * 2.3: Load the data

3. Data understanding(Exploratory Data Analysis (EDA))
   * 3.1: Exploring missing data
   * 3.2: Ditribution between features

4. Feature Engineering and Data Processing
   * 4.1: Handling features
   * 4.2: Handling Missing Data
   * 4.3: Converting Features

5. Model building
   * Decision Tree
   * Random Forest
   * Logistic Regression
   * k-Nearest Neighbors
   * Gaussian Naive Bayes
   * Support Vector Machines
   * Voting Classifier

6. Model Evaluation
   * 6.1: K-Fold Cross Validation
   * 6.2: Feature importance
   * 6.3: Hyperparameter Tuning
   * 6.4: Further evaluation

7. Conclusion


#### **1. Problem understanding and definition**
In this challenge, we need to complete the analysis of what sorts of people were most likely to survive. In particular, we apply the tools of machine learning to predict which passengers survived the tragedy.

##### **About the combined:**

There are three files in the combined: (1) train.csv, (2) test.csv, and (3) gender_submission.csv.

1. `train.csv:` contains the details of a subset of the passengers on board (891 passengers, to be exact -- where each passenger gets a different row in the table). The values in the second column ("Survived") can be used to determine whether each passenger survived or not:
- if it's a "1", the passenger survived.
- if it's a "0", the passenger died.
For instance, the first passenger listed in train.csv is Mr. Owen Harris Braund. He was 22 years old when he died on the Titanic.

2. `test.csv:` using the patterns we find in train.csv, we have to predict whether the other 418 passengers on board (in test.csv) survived.

3. `gender_submission.csv:` file is provided as an example that shows how you should structure your predictions. It predicts that all female passengers survived, and all male passengers died. Your hypotheses regarding survival will probably be different.
- a "PassengerId" column containing the IDs of each passenger from test.csv.
- a "Survived" column (that you will create!) with a "1" for the rows where you think the passenger survived, and a "0" where you predict that the passenger died.

The goal is to have a prediction file with exactly 2 columns:
- PassengerId (sorted in any order)
- Survived (contains your binary predictions: 1 for survived, 0 for deceased)

#### **2. Data collection and preparation**

##### **2.1: Loading the data files**    
Here we import the data. For this analysis, we will be exclusively working with the Training set. We will be validating based on data from the training set as well. For our final submissions, we will make predictions based on the test set.

In [ ]:
# data analysis and data wrangling
import numpy as np # Linear algebra
import pandas as pd # Data manipulation and analysis
# Data visualization
import seaborn as sns
import matplotlib.pyplot as plt
# Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
# Model Wrangling
from sklearn.model_selection import cross_val_score, GridSearchCV

##### **2.2: Data Description**    

The data has been split into two groups:
- **training set (train.csv)**    
The training set includes passengers survival status (also know as the ground truth from the titanic tragedy) which along with other features like gender, class, fare and pclass (passenger class) is used to create the machine learning model.
- **test set (test.csv)**    
The test set should be used to see how well the model performs on unseen data. The test set does not provide passengers survival status. We are going to use our model to predict passenger survival status.

##### **2.3: Load the data**

In [ ]:
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/titanic/test.csv")

**Inspect the first 3 rows for both train and test set**

In [ ]:
train_df.head(3)

In [ ]:
test_df.head(3)

#### **3. Data understanding using Exploratory Data Analysis (EDA)**
Exploratory Data Analysis refers to the critical process of performing initial investigations on data so as to discover patterns, to spot anomalies, to test hypothesis and to check assumptions with the help of summary statistics and graphical representations. In summary, it's an approach to analyzing data sets to summarize their main characteristics, often with visual methods.

**Check the columns**

In [ ]:
train_df.columns

**The training-set has 891 rows and 11 features + the target variable (survived).**

In [ ]:
train_df.shape

**What is the distribution of numerical feature values across the samples?**

This helps us determine, among other early insights, how representative is the training data of the actual problem domain.

- Total samples are 891 or 40% of the actual number of passengers on board the Titanic (2,224).
- Survived is a categorical feature with 0 or 1 values.
- Around 38% samples survived representative of the actual survival rate at 32%.
- Most passengers (> 75%) did not travel with parents or children.
- Nearly 30% of the passengers had siblings and/or spouse aboard.
- Fares varied significantly with few passengers (<1%) paying as high as $512.
- Few elderly passengers (<1%) within age range 65-80.

In [ ]:
train_df.describe()

**What is the distribution of categorical features?**

- Names are unique across the data (count=unique=891)
- Sex variable as two possible values with 65% male (top=male, freq=577/count=891).
- Cabin values have several dupicates across samples. Alternatively several passengers shared a cabin.
- Embarked takes three possible values. S port used by most passengers (top=S)
- Ticket feature has high ratio (22%) of duplicate values (unique=681).

In [ ]:
train_df.describe(include='O')

**Which features are categorical?**

- **Categorical**: Survived, Sex, and Embarked. **Ordinal**: Pclass.

**Which features are numerical?**

- **Continous**: Age, Fare. **Discrete**: SibSp, Parch.

**Which features are mixed data types?**

- Ticket is a mix of numeric and alphanumeric data types. Cabin is alphanumeric.

**Which features may contain errors or typos?**

- Name feature may contain errors or typos as there are several ways used to describe a name including titles, round brackets, and quotes used for alternative or short names.

**Which features contain blank, null or empty values?**

- Cabin > Age > Embarked features contain a number of null values in that order for the training data.
- Cabin > Age are incomplete in case of test data.

**What are the data types for various features?**

- Seven features are integer or floats. Six in case of test data.
- Five features are strings (object).

In [ ]:
train_df.info()

##### **3.1: Exploring missing data**

In [ ]:
train_df.isnull().sum()

The 'Embarked' feature has only 2 missing values, which can easily be filled or dropped. It will be much more tricky to deal with the ‘Age’ feature, which has 177 missing values. The ‘Cabin’ feature needs further investigation, but it looks like that we might want to drop it from the data since 77% is missing.

**Check for duplicated rows**

In [ ]:
train_df.duplicated().sum()

**"The captain goes down with the ship"**

it is a maritime tradition that a sea captain holds ultimate responsibility for both his/her ship and everyone embarked on it, and that in an emergency, he/she will either save them or die trying. In this case, Captain Edward Gifford Crosby went down with Titanic in a heroic gesture trying to save the passengers.

In [ ]:
train_df[train_df['Name'].str.contains('Capt')]

##### **3.2: Ditribution between features**

In [ ]:
# look at numeric and categorical values separately 
df_num = train_df[['Age','SibSp','Parch','Fare']]
df_cat = train_df[['Survived','Pclass','Sex','Ticket','Cabin','Embarked']]

In [ ]:
for dnum in df_num.columns:
    plt.hist(df_num[dnum])
    plt.title("The distributions for all numeric variables")
    plt.xlabel(dnum)
    plt.ylabel("Counts")
    plt.show()

In [ ]:
df_cat['Sex'].value_counts().index

In [ ]:
for dcat in df_cat.columns:
    sns.barplot(df_cat[dcat].value_counts().index,df_cat[dcat].value_counts()).set_title(dcat)
    plt.show()

In [ ]:
train_df.groupby(['Sex']).mean()

As previously mentioned, women are much more likely to survive than men. 74% of the women survived, while only 18% of men survived.

**Looking deeper into differences between females and males statistics**

In [ ]:
train_df.groupby(['Sex','Pclass']).mean()

We are grouping passengers based on Sex and Ticket class (Pclass). Notice the difference between survival rates between men and women. Women are much more likely to survive than men, specially women in the first and second class. It also shows that men in the first class are almost 3-times more likely to survive than men in the third class.

**Analyzing featurs with pivot table**

In [ ]:
train_df[['Pclass', 'Survived']].groupby(['Pclass'], as_index=False).mean().sort_values(by='Survived', ascending=False)

In [ ]:
train_df[["Sex", "Survived"]].groupby(['Sex'], as_index=False).mean().sort_values(by='Survived', ascending=False)

**Comparing survival with each of these categorical variables**

In [ ]:
pd.pivot_table(train_df, index = 'Survived', columns = 'Pclass', values = 'Ticket' ,aggfunc ='count')

In [ ]:
pd.pivot_table(train_df, index = 'Survived', columns = 'Sex', values = 'Ticket' ,aggfunc ='count')

In [ ]:
pd.pivot_table(train_df, index = 'Survived', columns = 'Embarked', values = 'Ticket' ,aggfunc ='count')

**Children below 18 years of age have higher chances of surviving.**

In [ ]:
train_df[train_df['Age']<18].groupby(['Sex','Pclass']).mean()

**Destribution of Age and Sex**

In [ ]:
survived = 'survived'
not_survived = 'not survived'
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 5))
women = train_df[train_df['Sex']=='female']
men = train_df[train_df['Sex']=='male']
# Plot Female Survived vs Not-Survived distribution
ax = sns.histplot(women[women['Survived']==1].Age.dropna(), bins=20, label = survived, ax = axes[0],color='b', kde=True)
ax = sns.histplot(women[women['Survived']==0].Age.dropna(), bins=20, label = not_survived, ax = axes[0],color='r', kde=True)
ax.legend()
ax.set_title('Female')
# Plot Male Survived vs Not-Survived distribution
ax = sns.histplot(men[men['Survived']==1].Age.dropna(), bins=20, label = survived, ax = axes[1],color='b', kde=True)
ax = sns.histplot(men[men['Survived']==0].Age.dropna(), bins=20, label = not_survived, ax = axes[1],color='r', kde=True)
ax.legend()
ax.set_title('Male');

We can see that:
- men have a higher probability of survival when they are between 18 and 35 years old. 
- For women, the survival chances are higher between 15 and 40 years old.
- For men the probability of survival is very low between the ages of 5 and 18, and after 35, but that isn’t true for women. 
- Another thing to note is that infants have a higher probability of survival.

**Distribution of Pclass and Survived**

In [ ]:
plt.subplots(figsize = (8,6))
sns.barplot(x='Pclass', y='Survived', data=train_df);
plt.title("Passenger Class Distribution - Survived Passengers", fontsize = 25)

In [ ]:
plt.subplots(figsize = (8,8))
ax=sns.countplot(x='Pclass',hue='Survived',data=train_df)
plt.title("Passenger Class Distribution - Survived vs Non-Survived", fontsize=20)

The graphs above clearly shows that economic status (Pclass) played an important role regarding the potential survival of the Titanic passengers. First class passengers had a much higher chance of survival than passengers in the 3rd class. We note that:
- 63% of the 1st class passengers survived the Titanic wreck
- 48% of the 2nd class passenger survived
- Only 24% of the 3rd class passengers survived

In [ ]:
sns.set(style='darkgrid')
plt.subplots(figsize = (8,6))
ax=sns.countplot(x='Sex', data = train_df, hue='Survived', edgecolor=(0,0,0), linewidth=2)
# Fixing title, xlabel and ylabel
plt.title('Passenger distribution of survived vs not-survived', fontsize=25)
plt.xlabel('Gender', fontsize=15)
plt.ylabel("# of Passenger Survived", fontsize = 15)
labels = ['Female', 'Male']
# Fixing xticks
plt.xticks(sorted(train_df.Survived.unique()),labels);

In [ ]:
plt.subplots(figsize=(10,8))
ax=sns.kdeplot(train_df.loc[(train_df['Survived'] == 0),'Pclass'],shade=True,color='r',label='Not Survived')
ax.legend()
ax=sns.kdeplot(train_df.loc[(train_df['Survived'] == 1),'Pclass'],shade=True,color='b',label='Survived')
ax.legend()
plt.title("Passenger Class Distribution - Survived vs Non-Survived", fontsize = 25)
labels = ['First', 'Second', 'Third']
plt.xticks(sorted(train_df.Pclass.unique()),labels);

**Correlation Matrix and Heatmap**

We need to convert some categorical values into numerical to see a better correlation. For now we will do it without converting.

In [ ]:
train_df.corr()['Survived'].sort_values(ascending=False)
corr = train_df.corr()
plt.figure(figsize=(15,10))
sns.heatmap(corr,annot=True)

We notice from the heatmap above that:
- Parents and sibling like to travel together (light blue squares)
- Age has a high negative correlation with number of siblings
- etc..

In [ ]:
train_df.corr()['Survived'].sort_values(ascending=False)

#### **4. Feature Engineering and Data Processing**

##### **4.1: Handling Features**

**Drop 'PassengerId'**    
First, I will drop ‘PassengerId’ from the train set, because it does not contribute to a persons' survival probability.

In [ ]:
train_df = train_df.drop("PassengerId",axis=1)
train_df.head(3)

**Combining SibSp and Parch**    
SibSp and Parch would make more sense as a combined feature that shows the total number of relatives a person has on the Titanic. I will create the new feature 'relative' below, and also a value that shows if someone is alone or not.

In [ ]:
combined = [train_df, test_df]
for data in combined:
    data['relative'] = data['SibSp'] + data['Parch']
    data.loc[data['relative'] > 0, 'isAlone'] = 0
    data.loc[data['relative'] == 0, 'isAlone'] = 1
    data['isAlone'] = data['isAlone'].astype(int)
train_df['isAlone'].value_counts()
train_df = train_df.drop(['Parch', 'SibSp'], axis=1)
test_df = test_df.drop(['Parch', 'SibSp'], axis=1)
train_df.head()

In [ ]:
train_df[['relative', 'Survived']].groupby(['relative'], as_index=False).mean().sort_values(by='Survived', ascending=False)

In [ ]:
train_df[['isAlone', 'Survived']].groupby(['isAlone'], as_index=False).mean()

In [ ]:
plt.subplots(figsize = (16,4))
ax = sns.lineplot(x='relative',y='Survived', data=train_df)

##### **4.2: Missing Data**

As a reminder, we have to deal with Cabin (687 missing values), Embarked (2 missing values) and Age (177 missing values).

**Cabin**     
We can either drop it or convert it to a new column that shows the deck of the cabin, we will convert the feature into a numeric variable. The missing values will be converted to zero. Note that the actual decks of the titanic, ranging from A to G.

In [ ]:
import re
deck = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "U": 8}
combined = [train_df, test_df]
for data in combined:
    data['Cabin'] = data['Cabin'].fillna("U0")
    data['Deck'] = data['Cabin'].map(lambda x: re.compile("([a-zA-Z]+)").search(x).group())
    data['Deck'] = data['Deck'].map(deck)
    data['Deck'] = data['Deck'].fillna(0)
    data['Deck'] = data['Deck'].astype(int)
# We can now drop the Cabin feature
train_df = train_df.drop(['Cabin'], axis=1)
test_df = test_df.drop(['Cabin'], axis=1)

**Age**    
We can fill this column with its mean()

In [ ]:
combined = [train_df, test_df]
mean = train_df['Age'].mean()
for data in combined:
    data['Age'].fillna(mean,inplace=True)

**Embarked**    
Since the Embarked feature has only 2 missing values, we will fill these with the most common one.

In [ ]:
train_df['Embarked'].describe()

We notice the most popular embark location is Southampton (S).

In [ ]:
common_value = 'S'
combined = [train_df, test_df]
for data in combined:
    data['Embarked'] = data['Embarked'].fillna(common_value)
train_df['Embarked'].isnull().sum()

#### Wrangle data

##### **4.3: Converting Features**

In [ ]:
train_df.info()
print("_"*50)
test_df.info()

We can see that 'Fare' and it is a float data-type. Also, we need to deal with 4 categorical features: Name, Sex, Ticket, and Embarked

**Fare:** Converting 'Fare' from float64 to int64 using the astype() function provided by pandas.

In [ ]:
combined = [train_df, test_df]
for data in combined:
    data['Fare'] = data['Fare'].fillna(0)
    data['Fare'] = data['Fare'].astype(int)
train_df.info()
print("_"*50)
test_df.info()

Let us create Age bands and determine correlations with Survived.

In [ ]:
# train_df['AgeBand'] = pd.cut(train_df['Age'], 5)
# train_df[['AgeBand', 'Survived']].groupby(['AgeBand'], as_index=False).mean().sort_values(by='AgeBand', ascending=True)
# # Let us replace Age with ordinals based on these bands.
# for data in combined:
#     data.loc[ data['Age'] <= 16, 'Age'] = 0
#     data.loc[(data['Age'] > 16) & (data['Age'] <= 32), 'Age'] = 1
#     data.loc[(data['Age'] > 32) & (data['Age'] <= 48), 'Age'] = 2
#     data.loc[(data['Age'] > 48) & (data['Age'] <= 64), 'Age'] = 3
#     data.loc[ data['Age'] > 64, 'Age']
# train_df.head()

**Change Age data type to int**

In [ ]:
data = [train_df, test_df]
for data in combined:
    data['Age'] = data['Age'].astype(int)
train_df.head()

**Name:** Feature Engineering the name of passengers to extract a person's title (Mr, Miss, Master, and Other), so we can build another feature called 'Title' out of it.

In [ ]:
combined = [train_df, test_df]
for data in combined:
    data['Title'] = data.Name.str.extract(' ([A-Za-z]+)\.', expand=False)
pd.crosstab(train_df['Title'], train_df['Sex'])

In [ ]:
titles = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Other": 5}
combined = [train_df, test_df]
for data in combined:
    # Extract titles
    data['Title'] = data.Name.str.extract('([A-Za-z]+)\.', expand=False)
    # Replace titles with a more common title or as Other
    data['Title'] = data['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr','Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Other')
    data['Title'] = data['Title'].replace('Mlle', 'Miss')
    data['Title'] = data['Title'].replace('Ms', 'Miss')
    data['Title'] = data['Title'].replace('Mme', 'Mrs')
    # Convert titles into numbers
    data['Title'] = data['Title'].map(titles)
    # Filling NaN with 0 just to be safe
    data['Title'] = data['Title'].fillna(0)
train_df = train_df.drop(['Name'], axis=1)
test_df = test_df.drop(['Name'], axis=1)
# Checking results
train_df.head()

**Sex:** Convert feature 'Sex' into numeric values.

In [ ]:
# train_df['Sex'] = train_df.Sex.map({'male':0,'female':1})
genders = {"male": 0, "female": 1}
combined = [train_df, test_df]
for data in combined:
    data['Sex'] = data['Sex'].map(genders)
train_df.head()

**Ticket:**

In [ ]:
train_df['Ticket'].describe()

Since the 'Ticket' feature has 681 unique values, it would be very hard to convert them into an useful feature. Hence, we will drop it from the DataFrame.

In [ ]:
train_df = train_df.drop(['Ticket'], axis=1)
test_df = test_df.drop(['Ticket'], axis=1)
train_df.head()

**Convert 'Embarked' feature into numeric values.**

In [ ]:
train_df['Embarked'].value_counts()

In [ ]:
combined = [train_df, test_df]
for data in combined:
    data['Embarked'] = data.Embarked.map({"S":0,"C":1,"Q":2})
train_df.head()

5. **Model building**

I will be using some of the most popular Machine Learning models in Data Science.

- Decision Tree
- Random Forest
- Logistic Regression
- k-Nearest Neighbors
- Gaussian Naive Bayes
- Support Vector Machines
- Voting Classifier

In [ ]:
X_train = train_df.drop("Survived", axis=1)
Y_train = train_df["Survived"]
X_test  = test_df.drop("PassengerId", axis=1).copy()
X_train.shape, Y_train.shape, X_test.shape

**The models are fitted and trained on the training set (seen/known data) and predicted on the testing set (unseen data)**

##### **Decision Tree**

This model uses a decision tree as a predictive model which maps features (tree branches) to conclusions about the target value (tree leaves). Tree models where the target variable can take a finite set of values are called classification trees; in these tree structures, leaves represent class labels and branches represent conjunctions of features that lead to those class labels. Decision trees where the target variable can take continuous values (typically real numbers) are called regression trees.

In [ ]:
dt = DecisionTreeClassifier()
dt.fit(X_train, Y_train)
Y_pred = dt.predict(X_test)

acc_dt = round(dt.score(X_train, Y_train) * 100, 2)
print("The accuracy is ==> ",round(acc_dt,2,), "%")

cv = cross_val_score(dt,X_train,Y_train,cv=10)
print("The cross-validation score is ==> ",cv)
print("The cv score mean is ==> ",cv.mean())

**Visualize the tree on selected features**

In [ ]:
from sklearn import tree
feature_names = ['Pclass','Sex','Age','Fare','relative']
X = train_df[feature_names].values
y = train_df['Survived'].values
# dt = DecisionTreeClassifier()
dt = DecisionTreeClassifier(max_depth=3,min_samples_leaf=2,max_leaf_nodes=10)
dt.fit(X, y)
data = tree.export_graphviz(
 dt,
 out_file='survivedTree.dot',
 feature_names=feature_names
)

##### **Random Forest**

Random forests or random decision forests are an ensemble learning method for classification, regression and other tasks, that operate by constructing a multitude of decision trees (n_estimators=100) at training time and outputting the class that is the mode of the classes (classification) or mean prediction (regression) of the individual trees.

In [ ]:
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, Y_train)
Y_pred = rf.predict(X_test)

acc_rf = round(rf.score(X_train, Y_train) * 100, 2)
print("The accuracy is ==> ",round(acc_rf,2,), "%")

cv = cross_val_score(rf,X_train,Y_train,cv=10)
print("The cross-validation score is ==> ",cv)
print("The cv score mean is ==> ",cv.mean())

##### **Logistic Regression**

Logistic regression measures the relationship between the categorical dependent variable (feature) and one or more independent variables (features) by estimating probabilities using a logistic function, which is the cumulative logistic distribution.

In [ ]:
lg = LogisticRegression(max_iter=300)
lg.fit(X_train, Y_train)
Y_pred = lg.predict(X_test)

acc_lg = round(lg.score(X_train, Y_train) * 100, 2)
print("The accuracy is ==> ",round(acc_lg,2,), "%")

cv = cross_val_score(lg,X_train,Y_train,cv=10)
print("The cross-validation score is ==> ",cv)
print("The cv score mean is ==> ",cv.mean())

In [ ]:
coeff_df = pd.DataFrame(train_df.columns.delete(0))
coeff_df.columns = ['Feature']
coeff_df["Correlation"] = pd.Series(lg.coef_[0])
coeff_df.sort_values(by='Correlation', ascending=False)

We can use Logistic Regression to validate our assumptions and decisions for feature creating and completing goals. This can be done by calculating the coefficient of the features in the decision function.

Positive coefficients increase the log-odds of the response (and thus increase the probability), and negative coefficients decrease the log-odds of the response (and thus decrease the probability).

- Sex is highest positivie coefficient, implying as the Sex value increases (male: 0 to female: 1), the probability of Survived=1 increases the most.
- Inversely as Pclass increases, probability of Survived=1 decreases the most.
- etc..

##### **k-Nearest Neighbors**

In pattern recognition, the k-Nearest Neighbors algorithm (or k-NN for short) is a non-parametric method used for classification and regression. A sample is classified by a majority vote of its neighbors, with the sample being assigned to the class most common among its k nearest neighbors (k is a positive integer, typically small). If k = 1, then the object is simply assigned to the class of that single nearest neighbor.

In [ ]:
knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(X_train, Y_train)
Y_pred = knn.predict(X_test)

acc_knn = round(knn.score(X_train, Y_train) * 100, 2)
print("The accuracy is ==> ",round(acc_knn,2,), "%")

cv = cross_val_score(knn,X_train,Y_train,cv=10)
print("The cross-validation score is ==> ",cv)
print("The cv score mean is ==> ",cv.mean())

##### **Gaussian Naive Bayes**

In machine learning, naive Bayes classifiers are a family of simple probabilistic classifiers based on applying Bayes' theorem with strong (naive) independence assumptions between the features. Naive Bayes classifiers are highly scalable, requiring a number of parameters linear in the number of variables (features) in a learning problem.

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, Y_train)
Y_pred = gnb.predict(X_test)

acc_gnb = round(gnb.score(X_train, Y_train) * 100, 2)
print("The accuracy is ==> ",round(acc_gnb,2,), "%")

cv = cross_val_score(gnb,X_train,Y_train,cv=10)
print("The cross-validation score is ==> ",cv)
print("The cv score mean is ==> ",cv.mean())

##### **Support Vector Machines**

Support Vector Machines is a supervised learning model with associated learning algorithms that analyze data used for classification and regression analysis. Given a set of training samples, each marked as belonging to one or the other of **two categories**, an SVM training algorithm builds a model that assigns new test samples to one category or the other, making it a non-probabilistic binary linear classifier.

In [ ]:
svc = SVC(probability=True)
svc.fit(X_train, Y_train)
Y_pred = svc.predict(X_test)

acc_svc = round(svc.score(X_train, Y_train) * 100, 2)
print("The accuracy is ==> ", round(acc_svc,2,), "%")

cv = cross_val_score(svc,X_train,Y_train,cv=10)
print("The cross-validation score is ==> ",cv)
print("The cv score mean is ==> ",cv.mean())

In [ ]:
gbc = GradientBoostingClassifier()
# gbc = GradientBoostingClassifier(ccp_alpha=0.0, criterion='friedman_mse', init=None,
#                            learning_rate=0.1, loss='log_loss', max_depth=3,
#                            max_features=None, max_leaf_nodes=None,
#                            min_impurity_decrease=0.0, min_samples_leaf=1,
#                            min_samples_split=2, min_weight_fraction_leaf=0.0,
#                            n_estimators=100, n_iter_no_change=None,
#                            random_state=123, subsample=1.0, tol=0.0001,
#                            validation_fraction=0.1, verbose=0,
#                            warm_start=False)

gbc.fit(X_train, Y_train)
Y_pred = gbc.predict(X_test)

acc_gbc = round(gbc.score(X_train, Y_train) * 100, 2)
print("The accuracy is ==> ", round(acc_gbc,2,), "%")

cv = cross_val_score(gbc,X_train,Y_train,cv=10)
print("The cross-validation score is ==> ",cv)
print("The cv score mean is ==> ",cv.mean())

##### **Voting Classifier**

Voting classifier takes all of the inputs and averages the results.
- For a "hard" voting classifier each classifier gets 1 vote "yes" or "no" and the result is just a popular vote. For this, you generally want odd numbers.
- A "soft" classifier averages the confidence of each of the models. If a the average confidence is > 50% that it is a 1 it will be counted as such.

In [ ]:
voting_clf = VotingClassifier(estimators = [('lg',lg),('knn',knn),('rf',rf),('gnb',gnb),('svc',svc),('dt',dt)], voting = 'soft')
voting_clf.fit(X_train,Y_train)
Y_pred = voting_clf.predict(X_test)

cv = cross_val_score(voting_clf,X_train,Y_train,cv=5)
print("The cross-validation score is ==> ",cv)
print("The cv score mean is ==> ",cv.mean())

#### **6. Model evaluation**

Which one is the best model?

We can now rank our evaluation of all the models to choose the best one for our problem. While both Decision Tree and Random Forest score the same, we choose to use Random Forest as they correct for decision trees' habit of overfitting to their training set. 

In [ ]:
models = pd.DataFrame({
    'Model': ['SVM', 'KNN', 'LG','RF', 'GNB', 'DT'],
    'Score': [acc_svc, acc_knn, acc_lg, acc_rf, acc_gnb,acc_dt]
    })
result_df = models.sort_values(by='Score', ascending=False).set_index('Score')
result_df.head(9)

The Random Forest classifier goes on top of the Machine Learning models, followed by Decision Tree and KNN respectfully. Now we need to check how the Random Forest performs by using cross validation.

##### **6.1: K-Fold Cross Validation**

K-Fold Cross Validation randomly splits the training data into K subsets called folds. Let's say we split our data into 4 folds (K = 4). The random forest model would be trained and validated 4 times, using a different fold for validation every time, while it would be trained on the remaining 3 folds.

The result of our K-Fold Cross Validation example would be an array that contains 4 different scores. We then need to compute the mean and the standard deviation for these scores.

The code below perform K-Fold Cross Validation on our random forest model, using 10 folds (K = 10). Therefore it outputs an array with 10 different scores.

In [ ]:
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, Y_train)
scores = cross_val_score(rf, X_train, Y_train, cv=10, scoring = "accuracy")
print("Scores:", scores)
print("Mean:", scores.mean())
print("Standard Deviation:", scores.std())

This looks much more realistic than before. The Random Forest classifier model has an average accuracy of 81% with a standard deviation of 3.7%. The standard deviation tell us how precise the estimates are.

This means the accuracy of our model can differ ± 3.7%. I believe the accuracy looks good. Since Random Forest is a model easy to use, we will try to increase its performance even further in the following section.

##### **6.2: Feature importance**

Another great quality of Random Forest is how easy it is to measure the relative importance of each feature. **Sklearn is able to measure the importance of a features by looking at how much the tree nodes that are used by that particular feature reduce impurity on average across all trees in the forest.** It computes this score automatically for each feature after training, and scales the results so that the sum of all importances is equal to 1.

In [ ]:
importances = pd.DataFrame({'feature':X_train.columns,'importance':np.round(rf.feature_importances_,3)})
importances = importances.sort_values('importance',ascending=False).set_index('feature')
importances.head(12)

In [ ]:
importances.plot.bar()

**Results**

'isAlone' and 'Embarked' don't play a significant role in the Random Forest classifiers prediction process. Thus, I will drop them from the DataFrame and train the classifier once again. We could also remove more features, however, this would inquire more investigations of the feature's effect on our model. For now, I will only remove those two from the DataFrame.

In [ ]:
# Dropping not_alone
train_df  = train_df.drop("isAlone", axis=1)
test_df  = test_df.drop("isAlone", axis=1)
# Dropping Parch
train_df  = train_df.drop("Embarked", axis=1)
test_df  = test_df.drop("Embarked", axis=1)

In [ ]:
# Reassigning features
X_train = train_df.drop("Survived", axis=1)
Y_train = train_df["Survived"]
X_test  = test_df.drop("PassengerId", axis=1).copy()

**Training the Random Forest classifier once again**

In [ ]:
rf = RandomForestClassifier(n_estimators=100, oob_score = True)
rf.fit(X_train, Y_train)
Y_prediction = rf.predict(X_test)
rf.score(X_train, Y_train)

acc_rf = round(rf.score(X_train, Y_train) * 100, 2)
print("The accuracy is ==> ", round(acc_rf,2,), "%")

cv = cross_val_score(rf,X_train,Y_train,cv=5)
print("The cross-validation score is ==> ",cv)
print("The cv score mean is ==> ",cv.mean())

**Feature importance without 'isAlone' and 'Embarked' features**

In [ ]:
importances = pd.DataFrame({'feature':X_train.columns,'importance':np.round(rf.feature_importances_,3)})
importances = importances.sort_values('importance',ascending=False).set_index('feature')
importances.head(12)

In [ ]:
importances.plot.bar()

The Random Forest model predicts as good as it did before. A general rule is that, the more features you have, the more likely your model will suffer from overfitting and vice versa.

##### **6.3: Hyperparameter Tuning**

In [ ]:
def clf_performance(classifier, model_name):
    print(model_name)
    print('Best Score: ' + str(classifier.best_score_))
    print('Best Parameters: ' + str(classifier.best_params_))

In [ ]:
# lr = LogisticRegression()
# param_grid = {'max_iter' : [2000],
#             'penalty' : ['l1', 'l2'],
#             'C' : np.logspace(-4, 4, 20),
#             'solver' : ['liblinear']}

# clf_lr = GridSearchCV(lg, param_grid = param_grid, cv = 5, verbose = True, n_jobs = -1)
# best_clf_lr = clf_lr.fit(X_train,Y_train)
# clf_performance(best_clf_lr,'Logistic Regression')

# svc = SVC(probability = True)
# param_grid = tuned_parameters = [{'kernel': ['rbf'], 'gamma': [.1,.5,1,2,5,10],
#                                 'C': [.1, 1, 10, 100, 1000]},
#                                 {'kernel': ['linear'], 'C': [.1, 1, 10, 100, 1000]},
#                                 {'kernel': ['poly'], 'degree' : [2,3,4,5], 'C': [.1, 1, 10, 100, 1000]}]
# clf_svc = GridSearchCV(svc, param_grid = param_grid, cv = 5, verbose = True, n_jobs = -1)
# best_clf_svc = clf_svc.fit(X_train,Y_train)
# clf_performance(best_clf_svc,'SVC')

# knn = KNeighborsClassifier()
# param_grid = {'n_neighbors' : [3,5,7,9],
#             'weights' : ['uniform', 'distance'],
#             'algorithm' : ['auto', 'ball_tree','kd_tree'],
#             'p' : [1,2]}
# clf_knn = GridSearchCV(knn, param_grid = param_grid, cv = 5, verbose = True, n_jobs = -1)
# best_clf_knn = clf_knn.fit(X_train,Y_train)
# clf_performance(best_clf_knn,'KNN')

rf = RandomForestClassifier(random_state = 1)
param_grid =  {'n_estimators': [400,450,500],
            'criterion':['gini','entropy'],
            'bootstrap': [True],
            'max_depth': [15, 20, 25],
            'max_features': ['auto','sqrt', 10],
            'min_samples_leaf': [2,3],
            'min_samples_split': [2,3]}
clf_rf = GridSearchCV(rf, param_grid = param_grid, cv = 5, verbose = True, n_jobs = -1)
best_clf_rf = clf_rf.fit(X_train,Y_train)

clf_performance(best_clf_rf,'Random Forest')

In [ ]:
best_rf = best_clf_rf.best_estimator_.fit(X_train,Y_train)
feat_importances = pd.Series(best_rf.feature_importances_, index=X_train.columns)
feat_importances.nlargest(20).plot(kind='barh')

**Testing best parameters**

We can use something called Out of Bag (OOB) score to estimate the generalization accuracy. Basically, the OOB score is computed as the number of correctly predicted rows from the out of the bag sample.

In [ ]:
rf = RandomForestClassifier(criterion = "gini", max_depth = 20,max_features='sqrt',min_samples_leaf = 3, min_samples_split = 2,n_estimators=450,oob_score=True, random_state=1, n_jobs=-1)
rf.fit(X_train, Y_train)
Y_pred = rf.predict(X_test)

rf.score(X_train, Y_train)
print("The oob score is ==> ", round(rf.oob_score_, 4)*100, "%")

cv = cross_val_score(rf,X_train,Y_train,cv=5)
print("The cross-validation score is ==> ",cv)
print("The cv score mean is ==> ",cv.mean())

##### **6.4: Further evaluation**

**Confusion Matrix**

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix
pred = cross_val_predict(rf, X_train, Y_train, cv=3)
confusion_matrix(Y_train, pred)

The first row is about the not-survived-predictions: 494 passengers were correctly classified as not survived (called true negatives) and 59 where wrongly classified as not survived (false positives).

The second row is about the survived-predictions: 93 passengers where wrongly classified as survived (false negatives) and 244 where correctly classified as survived (true positives).

A confusion matrix produces an idea of how accurate the model is.

**Precision and Recall**

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
print("Precision:", precision_score(Y_train, pred))
print("Recall:",recall_score(Y_train, pred))
print("F1-score:",f1_score(Y_train, pred))

Our model predicts correctly that a passenger survived 81% of the time (precision). The recall tells us that 73% of the passengers tested actually survived.

It is possible to combine precision and recall into one score, which is called the F-score. The F-score is computed with the harmonic mean of precision and recall.

There we have it, a 76% F-score. The score is not high because we have a recall of 71%. Unfortunately, the F-score is not perfect, because it favors classifiers that have a similar precision and recall. This can be a problem because often times we are searching for a high precision and other times a high recall. An increase of precision can result in a decrease of recall, and vice versa (depending on the threshold). This is called the **precision/recall trade-off**.

**Precision Recall Curve**

For each person the Random Forest algorithm has to classify, it computes a probability based on a function and it classifies the person as survived (when the score is bigger than threshold) or as not survived (when the score is smaller than the threshold). That’s why the threshold plays an important part in this process.

Let's plot the precision and recall with the threshold using matplotlib.

In [ ]:
from sklearn.metrics import precision_recall_curve
# Getting the probabilities of our predictions
y_scores = rf.predict_proba(X_train)
y_scores = y_scores[:,1]

precision, recall, threshold = precision_recall_curve(Y_train, y_scores)

def plot_precision_and_recall(precision, recall, threshold):
    plt.plot(threshold, precision[:-1], "r", label="precision", linewidth=5)
    plt.plot(threshold, recall[:-1], "b", label="recall", linewidth=5)
    plt.xlabel("threshold", fontsize=19)
    plt.legend(loc="upper right", fontsize=19)
    plt.ylim([0, 1])

plt.figure(figsize=(14, 7))
plot_precision_and_recall(precision, recall, threshold)

We can see in the graph above that the recall is falling of rapidly when the precision reaches around 85%. Thus, we may want to select the precision/recall trade-off before this point (maybe at around 75%).

Now we are able to choose a threshold, that gives the best precision/recall trade-off for the current problem. For example, if a precision of 80% is required, we can easily look at the plot and identify the threshold needed, which is around 0.4. Then we could train the model with exactly that threshold and expect the desired accuracy.

**Another way is to plot the precision and recall against each other:**

In [ ]:
def plot_precision_vs_recall(precision, recall):
    plt.plot(recall, precision, "b--", linewidth=3)
    plt.xlabel("precision", fontsize=19)
    plt.ylabel("recall", fontsize=19)
    plt.axis([0, 1.2, 0, 1.4])

plt.figure(figsize=(14, 7))
plot_precision_vs_recall(precision, recall)

**ROC AUC Curve**

Another way to evaluate and compare binary classifiers is the ROC AUC Curve. This curve plots the true positive rate (also called recall) against the false positive rate (ratio of incorrectly classified negative instances), instead of plotting the precision versus the recall values.

In [ ]:
from sklearn.metrics import roc_curve
# Compute true positive rate and false positive rate
false_positive_rate, true_positive_rate, thresholds = roc_curve(Y_train, y_scores)
# Plotting them against each other
def plot_roc_curve(false_positive_rate, true_positive_rate, label=None):
    plt.plot(false_positive_rate, true_positive_rate, linewidth=3, label=label)
    plt.plot([0, 1], [0, 1], 'r', linewidth=4)
    plt.axis([0, 1, 0, 1])
    plt.xlabel('False Positive Rate (FPR)', fontsize=16)
    plt.ylabel('True Positive Rate (TPR)', fontsize=16)
plt.figure(figsize=(14, 7))
plot_roc_curve(false_positive_rate, true_positive_rate)

The red line represents a purely random classifier (e.g. a coin flip). Thus, the classifier should be as far away from it as possible. The Random Forest model looks good.

There's a tradeoff here because the classifier produces more false positives the higher the true positive rate is.

**ROC AUC Score**

The ROC AUC Score is the corresponding score to the ROC AUC Curve. It is simply computed by measuring the area under the curve, which is called AUC.

A classifier that is 100% correct would have a ROC AUC Score of 1, and a completely random classifier would have a score of 0.5.

In [ ]:
from sklearn.metrics import roc_auc_score
r_a_score = roc_auc_score(Y_train, y_scores)
print("ROC-AUC-Score:", r_a_score)

**7: Conclusion**

We started this project by doing exploratory data analysis (EDA) where we got a feeling for the data, checked missing data, and learned which features are important. During this process we used seaborn and matplotlib to do visualizations and understand better the data. During the Feature Engineering and Data Processing part we computed missing values, converted features into numeric ones, grouped values into categories, and created new features. Afterwards, we trained 7 different machine learning models, picked the best of them (Random Forest), and applied cross validation on the model. Then, we discussed how Random Forest works, took a look at the importance it assigns to the different features, and tuned its performace through optimizing hyperparameter values. Lastly, we looked at Confusion Matrix and computed the models precision, recall and F-score.

Below you can see the final train_df DataFrame:

In [ ]:
train_df.head()

In [ ]:
submission = pd.DataFrame({
        "PassengerId": test_df["PassengerId"],
        "Survived": Y_pred
    })
submission.to_csv('submission.csv', index=False)

This was my first submission to the competition site Kaggle. Any suggestions to improve my scores are most welcome.

In [ ]:
gs = pd.read_csv("submission.csv")
gs.head()